# 04 · Grounding, hallucination, token budgets and the capstone

*Vision-Language Models (VLMs)*, notebook 4 of 4

A vision-language model can produce fluent, confident answers without looking at the right evidence, or without looking at the image at all. This notebook turns the model from notebook 3 into an object of study. You slice its accuracy, build counterfactual images that change exactly one attribute, catch a model that is right for the wrong reason, measure what it says about images with nothing in them, and work out how many tokens images and video frames cost. It ends with the scaffold for your capstone project.

## What you will build

- `accuracy_by_group`: accuracy for every slice of a test set;
- `make_counterfactual` and `counterfactual_scores`: images that differ in one attribute, and a check that the answers change with them;
- a shortcut model whose test accuracy is perfect but whose counterfactual accuracy is near zero;
- a hallucination check on images without a bar, and a model that learns to answer "none";
- `visual_token_budget` and `uniform_frame_indices`: the token cost of images and frames, and a frame sampler that can miss a brief event;
- a capstone report that records the data, the input path, the budget, the results and the limits.

## Learning objectives

By the end of this notebook you will be able to:

1. report accuracy by question type and by answer, and explain what an aggregate hides;
2. build counterfactual images and measure whether answers follow the visual evidence;
3. detect a shortcut that an in-distribution test set cannot reveal;
4. measure hallucination on images that lack the asked-about object, and train an abstention answer;
5. compute visual-token budgets for images and video frames, and explain how frame sampling limits what a model can observe;
6. assemble a capstone report whose claims match what was measured.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy and Matplotlib only, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded. The models are retrained from scratch with the code from notebooks 2 and 3, which takes a few seconds.
- Training uses the tiny tensor engine from Base Model Training, defined in a hidden recap cell.
- Every random number comes from a seeded generator, so your numbers match the text up to tiny floating-point differences.

## Setup

Run this cell first. It imports NumPy, `time` and Matplotlib and defines `practice`, the helper that checks each function you write.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below defines the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (a NumPy array that remembers how it was computed, so that `backward()` can fill in `.grad`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it is gradient-checked. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The hidden cell below re-defines the pieces from notebooks 1 to 3 under their usual names: the scene generator and the three sets of scenes and images, the preprocessing and the image encoder, the text encoder and the contrastive losses, the alignment model, and the whole question-answering path: `make_vqa_batch`, `project_visual_tokens`, `concat`, `build_input_sequence`, `answer_loss_mask`, `read_answers`, the small language model, `new_vqa_model`, `train_vqa`, `answer_questions` and `vqa_report`. Run it; open it only if you want to look back at the code.

In [ ]:
def _fresh(value):
    """A copy of every Tensor inside value, so each check starts from clean gradients."""
    if isinstance(value, Tensor):
        return Tensor(value.data.copy(), requires_grad=value.requires_grad)
    if isinstance(value, dict):
        return {key: _fresh(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_fresh(item) for item in value)
    return value


def _trainable_leaves(value):
    if isinstance(value, Tensor):
        return [value] if value.requires_grad else []
    if isinstance(value, dict):
        return [leaf for key in sorted(value) for leaf in _trainable_leaves(value[key])]
    if isinstance(value, (list, tuple)):
        return [leaf for item in value for leaf in _trainable_leaves(item)]
    return []


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """practice() for functions that build Tensors.

    Each case runs on fresh copies of its Tensor arguments. The check compares
    the values your function returns AND the gradients that flow back to every
    argument with requires_grad=True, so a function that cuts the graph (for
    example by working on .data) is caught as well.
    """
    def checked(function):
        def run(*args):
            copies = [_fresh(arg) for arg in args]
            out = function(*copies)
            outs = out if isinstance(out, tuple) else (out,)
            if not all(isinstance(item, Tensor) for item in outs):
                raise TypeError("return Tensor results built from Tensor operations, so gradients can flow")
            total = Tensor(0.0)
            for index, item in enumerate(outs):
                weights = np.random.default_rng(index).normal(size=item.shape)
                total = total + (item * weights).sum()
            total.backward()
            grads = [leaf.grad if leaf.grad is not None else np.zeros_like(leaf.data)
                     for leaf in _trainable_leaves(copies)]
            return {"values": [item.data for item in outs], "gradients": grads}
        return run

    mine, ref = checked(yours), checked(reference)
    return yours if practice(name, mine, ref, cases, atol) is mine else reference


COLOURS = ("red", "green", "blue", "yellow")
SIZES = ("small", "large")
ORIENTATIONS = ("horizontal", "vertical")
ATTRIBUTES = {"colour": COLOURS, "size": SIZES, "orientation": ORIENTATIONS}
RGB = np.array([[0.85, 0.12, 0.10],   # red
                [0.15, 0.70, 0.20],   # green
                [0.20, 0.30, 0.90],   # blue
                [0.90, 0.80, 0.10]])  # yellow
BAR = {0: (2, 8), 1: (3, 12)}  # size index -> (thickness, length) in pixels
IMAGE_SIZE = 16

# Every (colour, size, orientation) combination, as attribute indices.
ALL_COMBOS = [(c, s, o) for c in range(4) for s in range(2) for o in range(2)]
# Four combinations never shown in training. Every single attribute value
# still appears in training; only these particular combinations are new.
HELD_OUT_COMBOS = [(0, 0, 1), (1, 1, 0), (2, 0, 0), (3, 1, 1)]
TRAIN_COMBOS = [combo for combo in ALL_COMBOS if combo not in HELD_OUT_COMBOS]


def bar_box(size, orientation):
    """(height, width) of the bar in pixels."""
    thickness, length = BAR[int(size)]
    return (thickness, length) if orientation == 0 else (length, thickness)


def sample_scenes(n, combos, seed):
    """Describe n images: attribute indices, the bar's top-left corner and a noise seed."""
    rng = np.random.default_rng(seed)
    chosen = np.array(combos)[rng.integers(0, len(combos), n)]
    scenes = {"colour": chosen[:, 0], "size": chosen[:, 1], "orientation": chosen[:, 2]}
    boxes = np.array([bar_box(s, o) for s, o in zip(scenes["size"], scenes["orientation"])])
    scenes["row"] = rng.integers(0, IMAGE_SIZE - boxes[:, 0] + 1)
    scenes["col"] = rng.integers(0, IMAGE_SIZE - boxes[:, 1] + 1)
    scenes["noise_seed"] = rng.integers(0, 2**31, n)
    return scenes


def render(scenes):
    """Draw every scene as a (16, 16, 3) RGB image with values in [0, 1]."""
    n = len(scenes["colour"])
    images = np.empty((n, IMAGE_SIZE, IMAGE_SIZE, 3))
    for i in range(n):
        rng = np.random.default_rng(int(scenes["noise_seed"][i]))
        image = 0.08 + 0.06 * rng.random((IMAGE_SIZE, IMAGE_SIZE, 3))  # dark, slightly noisy background
        shade = rng.uniform(0.85, 1.15)  # brightness varies from image to image
        tint = rng.uniform(-0.1, 0.1, 3)  # and so does the exact hue
        colour = RGB[scenes["colour"][i]] * shade + tint
        height, width = bar_box(scenes["size"][i], scenes["orientation"][i])
        r, c = scenes["row"][i], scenes["col"][i]
        image[r:r + height, c:c + width] = colour + 0.04 * rng.normal(size=(height, width, 3))
        images[i] = np.clip(image, 0.0, 1.0)
    return images


def describe(scenes):
    """The caption of every scene, for example 'a large red horizontal bar'."""
    return [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar"
            for c, s, o in zip(scenes["colour"], scenes["size"], scenes["orientation"])]


def combo_ids(scenes):
    """One number from 0 to 15 per scene: which of the 16 combinations it shows."""
    return scenes["colour"] * 4 + scenes["size"] * 2 + scenes["orientation"]


train_scenes = sample_scenes(800, TRAIN_COMBOS, seed=101)
seen_scenes = sample_scenes(300, TRAIN_COMBOS, seed=202)        # new images, combinations seen in training
held_out_scenes = sample_scenes(300, HELD_OUT_COMBOS, seed=303)  # new images, combinations never seen
train_images, seen_images, held_out_images = render(train_scenes), render(seen_scenes), render(held_out_scenes)


def channel_stats(images):
    """Mean and standard deviation of each colour channel, over every pixel of every image."""
    return images.mean(axis=(0, 1, 2)), images.std(axis=(0, 1, 2))


def normalise_images(images, mean, std):
    """(images - mean) / std for every channel. images: (n, H, W, 3); mean, std: (3,)."""
    return (images - mean) / std


def patchify(images, patch_size):
    """Cut images (n, H, W, C) into non-overlapping square patches: (n, N, P*P*C), N = (H/P)*(W/P).

    Patches are numbered row by row; inside a patch the values run row by row,
    then column by column, then channel by channel.
    """
    n, height, width, channels = images.shape
    p = patch_size
    grid = images.reshape(n, height // p, p, width // p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, cols, p, p, C)
    return grid.reshape(n, (height // p) * (width // p), p * p * channels)


def unpatchify(patches, patch_size, height, width):
    """The inverse of patchify: (n, N, P*P*C) back to images (n, H, W, C)."""
    n, _, patch_dim = patches.shape
    p = patch_size
    channels = patch_dim // (p * p)
    grid = patches.reshape(n, height // p, width // p, p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, p, cols, p, C)
    return grid.reshape(n, height, width, channels)


def prepare(images, mean, std, patch_size=4):
    """Normalise with training statistics, then cut into patches."""
    return patchify(normalise_images(images, mean, std), patch_size)


def init_image_encoder(seed, patch_dim=48, n_patches=16, width=32):
    """Weights of the tiny image encoder, all trainable Tensors."""
    rng = np.random.default_rng(seed)
    return {
        "W_patch": parameter(patch_dim, width, rng=rng),
        "b_patch": zeros_parameter(width),
        "pos": parameter(n_patches, width, rng=rng, scale=0.1),
        "W_mix": parameter(width, width, rng=rng),
        "b_mix": zeros_parameter(width),
    }


def embed_patches(patches, W, b, pos):
    """Linear patch embedding plus one learned position vector per patch.

    patches: Tensor (n, N, patch_dim); W: (patch_dim, D); b: (D,); pos: (N, D).
    Returns a Tensor (n, N, D).
    """
    return patches @ W + b + pos


def encode_images(patches, params):
    """The tiny encoder: embed the patches, apply a ReLU layer to each patch, then average.

    patches: Tensor (n, N, patch_dim); params: the dictionary from init_image_encoder.
    Returns (features, pooled): Tensors of shape (n, N, D) and (n, D).
    """
    tokens = embed_patches(patches, params["W_patch"], params["b_patch"], params["pos"])
    features = (tokens @ params["W_mix"] + params["b_mix"]).relu()
    return features, features.mean(axis=1)


class Adam:
    """The Adam optimiser for a list of Tensors, as in frameworks such as PyTorch."""

    def __init__(self, params, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        self.params, self.lr, self.betas, self.eps = list(params), lr, betas, eps
        self.m = [np.zeros_like(p.data) for p in self.params]  # running mean of gradients
        self.v = [np.zeros_like(p.data) for p in self.params]  # running mean of squared gradients
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        beta1, beta2 = self.betas
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = beta1 * m + (1 - beta1) * p.grad
            v[...] = beta2 * v + (1 - beta2) * p.grad ** 2
            m_hat, v_hat = m / (1 - beta1 ** self.t), v / (1 - beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


VOCAB = ["<pad>", "a", "bar"] + list(SIZES) + list(COLOURS) + list(ORIENTATIONS)
TOKEN_ID = {word: index for index, word in enumerate(VOCAB)}
CAPTION_LENGTH = 6  # the longest caption has 5 words; the rest is padding


def tokenise(captions, length=CAPTION_LENGTH):
    """Word ids, padded with 0 (<pad>) to `length`: an int array (n, length)."""
    ids = np.zeros((len(captions), length), dtype=int)
    for i, caption in enumerate(captions):
        words = caption.split()
        ids[i, :len(words)] = [TOKEN_ID[word] for word in words]
    return ids


def encode_text(token_ids, E, W):
    """Bag-of-tokens text encoder: average the embeddings of the real (non-pad) tokens, then map.

    token_ids: int array (n, L) with 0 for <pad>; E: Tensor (V, D); W: Tensor (D, D_out).
    Returns a Tensor (n, D_out).
    """
    mask = (token_ids != 0).astype(float)[..., None]  # (n, L, 1): 1.0 for real tokens
    mean = (E[token_ids] * mask).sum(axis=1) / mask.sum(axis=1)  # (n, D)
    return mean @ W


def l2_normalise(x, eps=1e-8):
    """Scale every row of x to unit length, so dot products become cosine similarities."""
    return x / ((x * x).sum(axis=-1, keepdims=True) + eps) ** 0.5


def clip_loss(logits):
    """Symmetric InfoNCE with the matching pair on the diagonal. logits: Tensor (n, n) -> scalar Tensor."""
    targets = np.arange(logits.shape[0])
    return (cross_entropy(logits, targets) + cross_entropy(logits.T, targets)) * 0.5


def positive_mask(caption_ids):
    """mask[i, j] = 1.0 when items i and j have the same caption, else 0.0. caption_ids: (n,) -> (n, n)."""
    ids = np.asarray(caption_ids)
    return (ids[:, None] == ids[None, :]).astype(float)


def multi_positive_loss(logits, positives):
    """Symmetric contrastive loss in which every 1.0 in `positives` counts as a correct match.

    logits: Tensor (n, n), images in rows and captions in columns; positives: (n, n) array.
    """
    image_to_text = -(logits.softmax(axis=1) * positives).sum(axis=1).log().mean()
    text_to_image = -(logits.softmax(axis=0) * positives).sum(axis=0).log().mean()
    return (image_to_text + text_to_image) * 0.5


def recall_at_k(scores, relevant, k):
    """Fraction of queries (rows) with at least one relevant candidate among their k best scores.

    scores: (n_queries, n_candidates); relevant: same shape, True where a candidate is a valid match.
    """
    top_k = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return float(np.take_along_axis(relevant, top_k, axis=1).any(axis=1).mean())


ALL_CAPTIONS = [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar" for c, s, o in ALL_COMBOS]


def init_alignment_model(seed, embed_dim=16):
    """An image encoder with a projection head, and a bag-of-tokens text encoder."""
    rng = np.random.default_rng(seed + 1)
    return {
        "image": init_image_encoder(seed),
        "W_image": parameter(32, embed_dim, rng=rng),
        "E": parameter(len(VOCAB), embed_dim, rng=rng, scale=0.5),
        "W_text": parameter(embed_dim, embed_dim, rng=rng),
    }


def model_parameters(model):
    """Every Tensor in a (nested) dictionary of parameters."""
    found = []
    for value in model.values():
        if isinstance(value, dict):
            found += model_parameters(value)
        elif isinstance(value, Tensor):
            found.append(value)
    return found


def embed_images(model, patches):
    _, pooled = encode_images(Tensor(patches), model["image"])
    return l2_normalise(pooled @ model["W_image"])


def embed_captions(model, token_ids):
    return l2_normalise(encode_text(token_ids, model["E"], model["W_text"]))


def train_alignment(model, patches, token_ids, caption_ids, steps=200, batch_size=64, lr=0.01,
                    temperature=0.1, multi_positive=True, seed=0):
    """Train both encoders with the contrastive loss; returns the loss of every step."""
    optimiser = Adam(model_parameters(model), lr)
    order = np.random.default_rng(seed)
    losses = []
    for step in range(steps):
        batch = order.choice(len(token_ids), batch_size, replace=False)
        logits = embed_images(model, patches[batch]) @ embed_captions(model, token_ids[batch]).T / temperature
        if multi_positive:
            loss = multi_positive_loss(logits, positive_mask(caption_ids[batch]))
        else:
            loss = clip_loss(logits)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        losses.append(float(loss.data))
    return losses


def retrieval_report(model, patches, scenes):
    """Image-to-text retrieval over the 16 possible captions: R@1, R@3 and per-attribute top-1 accuracy."""
    scores = embed_images(model, patches).data @ embed_captions(model, tokenise(ALL_CAPTIONS)).data.T
    relevant = combo_ids(scenes)[:, None] == np.arange(len(ALL_CAPTIONS))[None, :]
    best = np.array(ALL_COMBOS)[scores.argmax(axis=1)]
    report = {"R@1": recall_at_k(scores, relevant, 1), "R@3": recall_at_k(scores, relevant, 3)}
    for index, attribute in enumerate(ATTRIBUTES):
        report[attribute] = float((best[:, index] == scenes[attribute]).mean())
    return report


LM_VOCAB = (["<pad>", "<end>", "what", "colour", "is", "the", "bar", "?", "how", "big", "it", "or", "none"]
            + list(COLOURS) + list(SIZES) + list(ORIENTATIONS))
LM_ID = {word: index for index, word in enumerate(LM_VOCAB)}
QUESTION_TYPES = ("colour", "size", "orientation")
QUESTIONS = {"colour": "what colour is the bar ?", "size": "how big is it ?",
             "orientation": "is the bar horizontal or vertical ?"}
TEXT_LENGTH = 9  # the longest question (7 tokens) + answer + <end>


def make_vqa_batch(scenes, seed, no_bar=None):
    """One question per scene. Returns (text, question_lengths, question_types, answers):

    text: int (n, TEXT_LENGTH) = question + answer + <end> + <pad>s; question_lengths: (n,);
    question_types: (n,) indices into QUESTION_TYPES; answers: (n,) ids of the correct answer word.
    Scenes flagged in `no_bar` get the answer "none".
    """
    rng = np.random.default_rng(seed)
    n = len(scenes["colour"])
    types = rng.integers(0, len(QUESTION_TYPES), n)
    text = np.zeros((n, TEXT_LENGTH), dtype=int)
    lengths, answers = np.zeros(n, dtype=int), np.zeros(n, dtype=int)
    for i in range(n):
        attribute = QUESTION_TYPES[types[i]]
        question = QUESTIONS[attribute].split()
        if no_bar is not None and no_bar[i]:
            answer = "none"
        else:
            answer = ATTRIBUTES[attribute][scenes[attribute][i]]
        ids = [LM_ID[word] for word in question] + [LM_ID[answer], LM_ID["<end>"]]
        text[i, :len(ids)] = ids
        lengths[i], answers[i] = len(question), LM_ID[answer]
    return text, lengths, types, answers


def decode(ids):
    return " ".join(LM_VOCAB[i] for i in ids if i != 0)


def project_visual_tokens(features, W, b):
    """Map vision features into the language model's width.

    features: Tensor (n, N, d_vision); W: (d_vision, d_model); b: (d_model,). Returns (n, N, d_model).
    """
    return features @ W + b


def concat(tensors, axis):
    """Join Tensors along an existing axis; the gradient is split back into the pieces."""
    tensors = [Tensor.lift(t) for t in tensors]
    sizes = [t.shape[axis] for t in tensors]
    data = np.concatenate([t.data for t in tensors], axis=axis)

    def backward(g):
        return tuple(np.split(g, np.cumsum(sizes)[:-1], axis=axis))

    return tensors[0]._result(data, tuple(tensors), backward, "concat")


def build_input_sequence(visual_tokens, text_ids, E, pos):
    """The language model's input: visual tokens first, then the embedded text, plus positions.

    visual_tokens: Tensor (n, N, d); text_ids: int (n, T); E: (V, d); pos: (N + T, d).
    Returns a Tensor (n, N + T, d).
    """
    return concat([visual_tokens, E[text_ids]], axis=1) + pos


def answer_loss_mask(question_lengths, n_visual, seq_len):
    """mask[i, t] = 1.0 where position t predicts row i's answer or its <end> token, else 0.0.

    Position t predicts the token at t + 1. The answer follows the question, so it is
    predicted from the last question token, at n_visual + question_length - 1.
    """
    positions = np.arange(seq_len)[None, :]
    last_question = n_visual + np.asarray(question_lengths)[:, None] - 1
    return ((positions == last_question) | (positions == last_question + 1)).astype(float)


def read_answers(logits, question_lengths, n_visual):
    """The model's answer: the most likely token at the position that predicts the answer.

    logits: array (n, L, V); question_lengths: (n,). Returns int ids (n,).
    """
    rows = np.arange(len(question_lengths))
    return logits[rows, n_visual + np.asarray(question_lengths) - 1].argmax(axis=-1)


def init_language_model(seed, n_visual, d_model=24, hidden=48, n_layers=2):
    """A small decoder-only transformer: embeddings, positions, two causal blocks and an output layer."""
    rng = np.random.default_rng(seed)
    lm = {"E": parameter(len(LM_VOCAB), d_model, rng=rng, scale=0.3),
          "pos": parameter(n_visual + TEXT_LENGTH - 1, d_model, rng=rng, scale=0.1)}
    for layer in range(n_layers):
        lm[f"block{layer}"] = {
            "ln1_g": Tensor(np.ones(d_model), requires_grad=True), "ln1_b": zeros_parameter(d_model),
            "Wq": parameter(d_model, d_model, rng=rng), "Wk": parameter(d_model, d_model, rng=rng),
            "Wv": parameter(d_model, d_model, rng=rng), "Wo": parameter(d_model, d_model, rng=rng, scale=0.1),
            "ln2_g": Tensor(np.ones(d_model), requires_grad=True), "ln2_b": zeros_parameter(d_model),
            "W1": parameter(d_model, hidden, rng=rng), "b1": zeros_parameter(hidden),
            "W2": parameter(hidden, d_model, rng=rng, scale=0.1),
        }
    lm["lnf_g"], lm["lnf_b"] = Tensor(np.ones(d_model), requires_grad=True), zeros_parameter(d_model)
    lm["W_out"] = parameter(d_model, len(LM_VOCAB), rng=rng, scale=0.1)
    return lm


def language_model_forward(lm, x, heads=2):
    """Next-token logits (n, L, V) for an input sequence x: Tensor (n, L, d)."""
    n, length, d = x.shape
    head_dim = d // heads
    causal = np.triu(np.full((length, length), -1e9), 1)  # no position may look ahead

    def split_heads(t):
        return t.reshape(n, length, heads, head_dim).transpose(0, 2, 1, 3)

    for name in sorted(key for key in lm if key.startswith("block")):
        block = lm[name]
        h = layer_norm(x, block["ln1_g"], block["ln1_b"])
        q, k, v = split_heads(h @ block["Wq"]), split_heads(h @ block["Wk"]), split_heads(h @ block["Wv"])
        weights = (q @ k.transpose(0, 1, 3, 2) / np.sqrt(head_dim) + causal).softmax(axis=-1)
        x = x + (weights @ v).transpose(0, 2, 1, 3).reshape(n, length, d) @ block["Wo"]
        h = layer_norm(x, block["ln2_g"], block["ln2_b"])
        x = x + (h @ block["W1"] + block["b1"]).relu() @ block["W2"]
    return layer_norm(x, lm["lnf_g"], lm["lnf_b"]) @ lm["W_out"]


def visual_inputs(model, patches):
    """What the connector receives: the aligned image embedding from notebook 2 as one token,
    or the encoder's patch features as one token per patch."""
    features, pooled = encode_images(Tensor(patches), model["vision"]["image"])
    if model["visual_tokens"] == "embedding":
        embedding = l2_normalise(pooled @ model["vision"]["W_image"])  # (n, 16)
        return embedding.reshape(embedding.shape[0], 1, embedding.shape[1])
    return features  # (n, 16 patches, 32)


def vqa_logits(model, patches, text_in):
    visual = project_visual_tokens(visual_inputs(model, patches), model["projector"]["W"], model["projector"]["b"])
    x = build_input_sequence(visual, text_in, model["language_model"]["E"], model["language_model"]["pos"])
    return language_model_forward(model["language_model"], x)


def new_vqa_model(alignment_model, seed, visual_tokens="embedding", freeze_vision=True, d_model=24):
    """A fresh projector and language model on top of a copy of the aligned image encoder."""
    def copy(t):
        return Tensor(t.data.copy(), requires_grad=not freeze_vision)

    n_visual, d_vision = (1, 16) if visual_tokens == "embedding" else (16, 32)
    rng = np.random.default_rng(seed)
    return {
        "visual_tokens": visual_tokens, "n_visual": n_visual,
        "vision": {"image": {key: copy(value) for key, value in alignment_model["image"].items()},
                   "W_image": copy(alignment_model["W_image"])},
        "projector": {"W": parameter(d_vision, d_model, rng=rng), "b": zeros_parameter(d_model)},
        "language_model": init_language_model(seed + 1, n_visual, d_model),
    }


def train_vqa(model, patches, text, question_lengths, steps=300, batch_size=32, lr=0.01, vision_lr=0.001, seed=0):
    """Answer-only training. The vision encoder is updated only if its Tensors require gradients."""
    new_parts = model_parameters(model["projector"]) + model_parameters(model["language_model"])
    optimisers = [Adam(new_parts, lr)]
    vision = [p for p in model_parameters(model["vision"]) if p.requires_grad]
    if vision:
        optimisers.append(Adam(vision, vision_lr))  # pretrained weights get a smaller step
    order = np.random.default_rng(seed)
    n_visual = model["n_visual"]
    seq_len = n_visual + TEXT_LENGTH - 1
    losses = []
    for step in range(steps):
        batch = order.choice(len(text), batch_size, replace=False)
        logits = vqa_logits(model, patches[batch], text[batch, :-1])
        targets = np.concatenate([np.zeros((batch_size, n_visual), dtype=int), text[batch]], axis=1)[:, 1:]
        loss = cross_entropy(logits, targets, answer_loss_mask(question_lengths[batch], n_visual, seq_len))
        for optimiser in optimisers:
            optimiser.zero_grad()
        loss.backward()
        for optimiser in optimisers:
            optimiser.step()
        losses.append(float(loss.data))
    return losses


def answer_questions(model, patches, text, question_lengths):
    """Predicted answer ids and the probability the model gave its answer."""
    logits = vqa_logits(model, patches, text[:, :-1]).data
    at = logits[np.arange(len(text)), model["n_visual"] + np.asarray(question_lengths) - 1]
    probs = np.exp(at - at.max(axis=-1, keepdims=True))
    probs /= probs.sum(axis=-1, keepdims=True)
    return read_answers(logits, question_lengths, model["n_visual"]), probs.max(axis=-1)


def vqa_report(model, patches, text, question_lengths, types, answers):
    """Accuracy for each question type and overall."""
    predicted, _ = answer_questions(model, patches, text, question_lengths)
    report = {name: float((predicted[types == t] == answers[types == t]).mean()) for t, name in enumerate(QUESTION_TYPES)}
    report["all"] = float((predicted == answers).mean())
    return report

### Retrain the model from notebook 3

The next cell repeats notebook 3's frozen-encoder run with the same seeds: align the encoders, then train the projector and the language model on top of the aligned image embedding. `ask` (given) puts the same question to every scene and returns the predicted and the correct answer ids.

In [ ]:
def ask(model, scenes, attribute, images=None, mean=None, std=None):
    """Ask every scene the same question about `attribute`; returns (predicted ids, true ids)."""
    words = QUESTIONS[attribute].split()
    n = len(scenes["colour"])
    text = np.zeros((n, TEXT_LENGTH), dtype=int)
    text[:, :len(words)] = [LM_ID[word] for word in words]
    lengths = np.full(n, len(words))
    images = render(scenes) if images is None else images
    predicted, _ = answer_questions(model, prepare(images, mean, std), text, lengths)
    truth = np.array([LM_ID[ATTRIBUTES[attribute][value]] for value in scenes[attribute]])
    return predicted, truth


def blank_images(n, seed):
    """Background only: the same dark noise as render() draws, with no bar."""
    rng = np.random.default_rng(seed)
    return np.clip(0.08 + 0.06 * rng.random((n, IMAGE_SIZE, IMAGE_SIZE, 3)), 0.0, 1.0)


mean, std = channel_stats(train_images)
train_patches, seen_patches, held_out_patches = (prepare(images, mean, std)
                                                 for images in (train_images, seen_images, held_out_images))
start = time.perf_counter()
alignment = init_alignment_model(seed=0)
train_alignment(alignment, train_patches, tokenise(describe(train_scenes)), combo_ids(train_scenes), seed=2)
train_text, train_lengths, train_types, train_answers = make_vqa_batch(train_scenes, seed=7)
model = new_vqa_model(alignment, seed=10, freeze_vision=True)
train_vqa(model, train_patches, train_text, train_lengths, seed=3)
seen_vqa, held_out_vqa = make_vqa_batch(seen_scenes, seed=8), make_vqa_batch(held_out_scenes, seed=9)
print(f"trained in {time.perf_counter() - start:.2f} s")
print("seen:    ", {key: round(value, 3) for key, value in vqa_report(model, seen_patches, *seen_vqa).items()})
print("held-out:", {key: round(value, 3) for key, value in vqa_report(model, held_out_patches, *held_out_vqa).items()})

## Part 1 · Accuracy by slice

One accuracy number averages over everything the test set contains. If a model answers almost every size question but fails a quarter of the colour questions about one combination, the average can still look good. Evaluation should break results down by the conditions that matter for the intended use: question type, answer value, object size, image quality, language, document layout, and so on. Here the natural slices are the question type and the correct answer.

Choose the metric to match the output, too. Our answers are single words, so exact match is right. Free-form answers need accepted-answer sets or human judgement, reading text in images (OCR) is usually scored with character or word error rates, and grounding to image regions with the overlap between predicted and annotated boxes.

In [ ]:
predicted, _ = answer_questions(model, held_out_patches, held_out_vqa[0], held_out_vqa[1])
held_out_correct = predicted == held_out_vqa[3]
print(f"held-out accuracy: {held_out_correct.mean():.3f} over {len(held_out_correct)} questions")

### ✍️ Your turn: `accuracy_by_group`

Write `accuracy_by_group(correct, groups)`.

- `correct`: a boolean array of shape (n,), True where an answer was right;
- `groups`: an array of shape (n,) of group labels, such as question types or answer words;
- return a dictionary that maps every distinct label to the accuracy within that group, as a Python `float`. Use plain Python labels as keys, in sorted order: `np.unique(groups).tolist()` gives exactly that.

*Hint:* for each label, `correct[groups == label].mean()`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def accuracy_by_group_reference(correct, groups):
    """Accuracy within each group. correct: (n,) bools; groups: (n,) labels. Returns {label: accuracy}."""
    correct, groups = np.asarray(correct, dtype=float), np.asarray(groups)
    return {label: float(correct[groups == label].mean()) for label in np.unique(groups).tolist()}

In [ ]:
def accuracy_by_group(correct, groups):
    """Accuracy within each group. correct: (n,) bools; groups: (n,) labels. Returns {label: accuracy}."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
accuracy_by_group = practice("accuracy_by_group", accuracy_by_group, accuracy_by_group_reference, cases=[
    (np.array([True, False, True, True]), np.array(["a", "a", "b", "b"])),
    (np.array([True, True, False]), np.array([2, 0, 2])),
    (np.array([False]), np.array(["only"])),  # edge case: one example in one group
])

In [ ]:
for name, patches, questions in (("seen", seen_patches, seen_vqa), ("held-out", held_out_patches, held_out_vqa)):
    text, lengths, types, answers = questions
    predicted, _ = answer_questions(model, patches, text, lengths)
    correct = predicted == answers
    by_type = accuracy_by_group(correct, np.array(QUESTION_TYPES)[types])
    by_answer = accuracy_by_group(correct, np.array(LM_VOCAB)[answers])
    print(f"{name:9s} by question: " + ", ".join(f"{key} {value:.2f}" for key, value in by_type.items()))
    print(f"{'':9s} by answer:   " + ", ".join(f"{key} {value:.2f}" for key, value in by_answer.items()))

**Expected behaviour.** ✅ on three cases. On seen combinations every slice is close to 1. On held-out combinations the slices show where the misses are. In our run colour questions are the weakest slice, at about 0.73, and the per-answer slices are sharper still: the held-out red bars, all small and vertical, got the right colour only about one time in five, and the blue ones about two times in three. An overall accuracy of 0.86 hid both. Some slices hold only a few dozen questions, so treat differences of a few percentage points as noise.

## Part 2 · Counterfactual images

An accuracy score says how often an answer was right, not *why*. A **counterfactual** test changes the evidence that should determine the answer, keeps everything else fixed, and checks that the answer changes with it. Because every image here is drawn from a scene record, a counterfactual is easy to make exactly: copy the scene, change one attribute, and render it again with the same position and the same noise. When a change of size or orientation would push the bar over the edge, `fit_inside` (given) moves it back inside the image.

For pairs of an original and a changed image, two numbers summarise the result:

- **both correct**: the fraction of pairs answered correctly on both images, so that the answer followed the change;
- **stuck**: the fraction of pairs whose answer stayed the same although the correct answer changed. A stuck answer cannot be grounded in the attribute that was changed.

In [ ]:
def fit_inside(scenes):
    """Move each bar back inside the image if its box would overflow the edge."""
    boxes = np.array([bar_box(s, o) for s, o in zip(scenes["size"], scenes["orientation"])])
    fitted = dict(scenes)
    fitted["row"] = np.minimum(scenes["row"], IMAGE_SIZE - boxes[:, 0])
    fitted["col"] = np.minimum(scenes["col"], IMAGE_SIZE - boxes[:, 1])
    return fitted


example = {key: value[:1] for key, value in held_out_scenes.items()}
variants = {"original": example}
for attribute in ATTRIBUTES:
    changed = dict(example)
    changed[attribute] = (example[attribute] + 1) % len(ATTRIBUTES[attribute])
    variants[attribute + " changed"] = fit_inside(changed)
fig, axes = plt.subplots(1, 4, figsize=(8, 2.4))
for ax, (label, scene) in zip(axes, variants.items()):
    ax.imshow(render(scene)[0], interpolation="nearest")
    ax.set_title(label + "\n" + " ".join(describe(scene)[0].split()[1:4]), fontsize=8)
    ax.axis("off")
plt.show()

### ✍️ Your turn: `make_counterfactual`

Write `make_counterfactual(scenes, attribute, seed)`.

- `scenes`: a scene dictionary; `attribute`: `"colour"`, `"size"` or `"orientation"`; `seed`: an integer;
- return a **new** dictionary in which `attribute` has a *different* value in every scene, chosen at random among the other values, and every other entry is kept. Finish with `fit_inside`. Do not modify the input dictionary or its arrays.

*Hints:* make the generator with `np.random.default_rng(seed)`. To move every value to a different one, add a random shift between 1 and $K - 1$ for an attribute with $K$ values and wrap around with `% K`: `rng.integers(1, K, n)`. `dict(scenes)` copies the dictionary but not the arrays, so build a new array for the changed attribute rather than writing into the old one.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def make_counterfactual_reference(scenes, attribute, seed):
    """A copy of `scenes` in which `attribute` has a different value in every scene.

    Everything else (position, noise, the other attributes) is kept, except that a bar
    whose new shape would overflow the image is moved back inside with fit_inside.
    The input dictionary is not modified.
    """
    rng = np.random.default_rng(seed)
    n_values = len(ATTRIBUTES[attribute])
    changed = dict(scenes)
    changed[attribute] = (scenes[attribute] + rng.integers(1, n_values, len(scenes[attribute]))) % n_values
    return fit_inside(changed)

In [ ]:
def make_counterfactual(scenes, attribute, seed):
    """A copy of `scenes` in which `attribute` has a different value in every scene.

    Everything else (position, noise, the other attributes) is kept, except that a bar
    whose new shape would overflow the image is moved back inside with fit_inside.
    The input dictionary is not modified.
    """
    # ✍️ Your code here (four lines)
    raise NotImplementedError

In [ ]:
original_colours = held_out_scenes["colour"].copy()
make_counterfactual = practice("make_counterfactual", make_counterfactual, make_counterfactual_reference, cases=[
    (held_out_scenes, "colour", 40),
    (held_out_scenes, "orientation", 41),
    ({key: value[:1] for key, value in seen_scenes.items()}, "size", 42),  # edge case: a single scene
])
assert np.array_equal(held_out_scenes["colour"], original_colours), "make_counterfactual must not change its input"
changed = make_counterfactual(held_out_scenes, "colour", seed=40)
print("colours before:", held_out_scenes["colour"][:8], " after:", changed["colour"][:8])

### ✍️ Your turn: `counterfactual_scores`

Write `counterfactual_scores(pred_original, pred_changed, truth_original, truth_changed)`.

- all four arguments: integer arrays of shape (n,) of answer ids, for the original images and their counterfactuals;
- return the dictionary `{"both_correct": ..., "stuck": ...}` with Python floats: the fraction of pairs right on both images, and the fraction of pairs whose prediction did not change although the truth did.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def counterfactual_scores_reference(pred_original, pred_changed, truth_original, truth_changed):
    """How answers follow a one-attribute change.

    both_correct: fraction of pairs answered correctly on both images.
    stuck: fraction of pairs whose answer stayed the same although the correct answer changed.
    """
    pred_original, pred_changed = np.asarray(pred_original), np.asarray(pred_changed)
    truth_original, truth_changed = np.asarray(truth_original), np.asarray(truth_changed)
    both = (pred_original == truth_original) & (pred_changed == truth_changed)
    stuck = (pred_original == pred_changed) & (truth_original != truth_changed)
    return {"both_correct": float(both.mean()), "stuck": float(stuck.mean())}

In [ ]:
def counterfactual_scores(pred_original, pred_changed, truth_original, truth_changed):
    """How answers follow a one-attribute change.

    both_correct: fraction of pairs answered correctly on both images.
    stuck: fraction of pairs whose answer stayed the same although the correct answer changed.
    """
    # ✍️ Your code here (three or four lines)
    raise NotImplementedError

In [ ]:
counterfactual_scores = practice("counterfactual_scores", counterfactual_scores, counterfactual_scores_reference, cases=[
    (np.array([1, 2, 3, 4]), np.array([1, 5, 3, 7]), np.array([1, 2, 0, 4]), np.array([2, 5, 0, 7])),
    (np.array([3, 3]), np.array([3, 3]), np.array([3, 3]), np.array([3, 4])),
    (np.array([0]), np.array([1]), np.array([0]), np.array([1])),  # edge case: one pair, answered correctly twice
])

In [ ]:
print(f"{'changed attribute':18s} {'accuracy':>9s} {'both correct':>13s} {'stuck':>7s}")
grounding = {}
for index, attribute in enumerate(ATTRIBUTES):
    changed = make_counterfactual(held_out_scenes, attribute, seed=40 + index)
    pred_original, truth_original = ask(model, held_out_scenes, attribute, held_out_images, mean, std)
    pred_changed, truth_changed = ask(model, changed, attribute, None, mean, std)
    grounding[attribute] = counterfactual_scores(pred_original, pred_changed, truth_original, truth_changed)
    accuracy = float((pred_original == truth_original).mean())
    print(f"{attribute:18s} {accuracy:9.3f} {grounding[attribute]['both_correct']:13.3f} {grounding[attribute]['stuck']:7.3f}")

**Expected behaviour.** ✅ on both type-alongs, and a table with one row per attribute, for the held-out scenes and their counterfactuals. Size answers follow the change almost always. Orientation has the highest stuck rate in our run, about 0.16: for roughly one pair in six, turning the bar did not change the answer, so for those images the orientation answer was not grounded in the bar's orientation. Note that a counterfactual can move an image *out of* the held-out combinations, so "both correct" mixes seen and unseen combinations; it measures whether the answer follows the change, not held-out accuracy.

Counterfactual tests have limits. Change too much, or produce something unrealistic, and a changed answer may be a reaction to an odd image rather than evidence of grounding. Our changes keep the image inside the training distribution, which is easy for generated data and hard for photographs, where image editing leaves traces.

## 🧪 Experiment 1: a shortcut that accuracy cannot see

Suppose the training data has a coincidence: every red or green bar is large, and every blue or yellow bar is small. A test set collected the same way has the same coincidence. The cell trains a model on such data, using the sixteen patch tokens of notebook 3's experiment as its visual input, and tests it twice: on new images from the same confounded distribution, and on counterfactuals that change **only the size**.

In [ ]:
CONFOUNDED_COMBOS = [(c, 1 if c in (0, 1) else 0, o) for c in range(4) for o in range(2)]  # size follows colour
confounded_train = sample_scenes(800, CONFOUNDED_COMBOS, seed=111)
confounded_test = sample_scenes(300, CONFOUNDED_COMBOS, seed=222)
confounded_text, confounded_lengths, _, _ = make_vqa_batch(confounded_train, seed=12)
shortcut_model = new_vqa_model(alignment, seed=20, visual_tokens="patches", freeze_vision=True)
start = time.perf_counter()
train_vqa(shortcut_model, prepare(render(confounded_train), mean, std), confounded_text, confounded_lengths, seed=4)
print(f"trained in {time.perf_counter() - start:.2f} s")

pred_original, truth_original = ask(shortcut_model, confounded_test, "size", None, mean, std)
pred_changed, truth_changed = ask(shortcut_model, make_counterfactual(confounded_test, "size", seed=43), "size", None, mean, std)
size_accuracy = float((pred_original == truth_original).mean())
shortcut = counterfactual_scores(pred_original, pred_changed, truth_original, truth_changed)
print(f"size accuracy on the confounded test set: {size_accuracy:.3f}")
print(f"size counterfactuals: both correct {shortcut['both_correct']:.3f}, stuck {shortcut['stuck']:.3f}")
assert size_accuracy >= 0.9, "on data with the same coincidence, the shortcut looks perfect"
assert shortcut["both_correct"] <= size_accuracy - 0.5, "the counterfactual test should expose the shortcut"

**Expected behaviour.** The size accuracy on the confounded test set is 1.0, yet when only the size changes the answers almost never follow: "both correct" is near 0 and "stuck" near 1. The model answers size questions by looking at the *colour*, which predicted size perfectly in its data. Across eight seeds, size accuracy was always 1.0 and "both correct" never above 0.06. No test set drawn from the same distribution could have revealed this.

The same idea built the VQA v2 benchmark: its authors collected, for many questions, a second, similar image with a different answer, so that a model relying on language priors or coincidences would fail. A strong *text-only* baseline is the language version of the same warning sign: if the question alone predicts the answer, high accuracy does not show that the image was used.

## Part 3 · Hallucination: images with no bar

What does the model say about a picture that has no bar at all? Every training question had an answer among the bar's attributes, so the model has no way to say "there is nothing there"; whatever it answers is invented. Object-hallucination benchmarks such as POPE ask models about objects that are absent from the image and count how often they claim the object is there. The cell asks our model all three questions about 150 blank backgrounds.

In [ ]:
n_blank = 150
blank_patches = prepare(blank_images(n_blank, seed=55), mean, std)
blank_text, blank_lengths, blank_types, _ = make_vqa_batch(sample_scenes(n_blank, TRAIN_COMBOS, seed=56), seed=57)
blank_predicted, blank_confidence = answer_questions(model, blank_patches, blank_text, blank_lengths)
invented = float((blank_predicted != LM_ID["none"]).mean())
print(f"answers that invent an attribute: {invented:.0%}; mean probability the model gave its answer: {blank_confidence.mean():.2f}")
print("most common answers:", {LM_VOCAB[i]: int(c) for i, c in zip(*np.unique(blank_predicted, return_counts=True))})

**Expected behaviour.** Every answer is an invented attribute, usually given with high probability: about 0.99 on average in our run. Confidence is no protection: the model has never been shown that "no bar" is possible.

## 🧪 Experiment 2: teach the model to abstain

The visual instruction tuning lesson recommends including unanswerable questions, with a clear answer policy. Here the policy is the answer `none`. The cell replaces 10% of the training images with blank backgrounds whose answer is `none`, trains a fresh model, and measures both sides of the trade: abstention on blank images, and accuracy on the held-out images, which all contain a bar.

In [ ]:
no_bar = np.random.default_rng(58).random(len(train_images)) < 0.1
abstain_images = train_images.copy()
abstain_images[no_bar] = blank_images(int(no_bar.sum()), seed=59)
abstain_text, abstain_lengths, _, _ = make_vqa_batch(train_scenes, seed=7, no_bar=no_bar)
abstain_model = new_vqa_model(alignment, seed=30, freeze_vision=True)
start = time.perf_counter()
train_vqa(abstain_model, prepare(abstain_images, mean, std), abstain_text, abstain_lengths, seed=5)
abstain_predicted, _ = answer_questions(abstain_model, blank_patches, blank_text, blank_lengths)
abstain_rate = float((abstain_predicted == LM_ID["none"]).mean())
abstain_held_out = vqa_report(abstain_model, held_out_patches, *held_out_vqa)
print(f"trained in {time.perf_counter() - start:.2f} s with {no_bar.sum()} blank training images")
print(f"blank images answered 'none': before {1 - invented:.0%}, after {abstain_rate:.0%}")
print("held-out accuracy after:", {key: round(value, 3) for key, value in abstain_held_out.items()})
assert invented >= 0.95, "without unanswerable examples the model invents an answer for blank images"
assert abstain_rate >= 0.8, "with them, it should learn to answer 'none'"
assert abstain_held_out["all"] >= 0.6, "abstaining must not cost the real questions"

**Expected behaviour.** 73 of the 800 training images become blank backgrounds. Afterwards the model answers `none` for every blank test image. Its held-out accuracy on images that do contain a bar was 0.90 in our run, against 0.86 for the original model; across eight seeds it ranged from 0.75 to 0.97, while the original model ranged from 0.86 to 0.98, so abstention training may cost a little held-out accuracy in some runs. Abstention on a blank background is the easiest possible case: real hallucination involves cluttered scenes, objects that are plausible but absent, and questions whose wording suggests an answer. Test those explicitly for any system you deploy.

## Part 4 · Token budgets for images and video

Every visual token takes a place in the language model's context, next to the text. For square patches of side $P$ on an $H \times W$ image and $F$ frames,

$$\text{visual tokens} = F \cdot \frac{H}{P} \cdot \frac{W}{P}, \qquad \text{attention entries} = (\text{visual tokens} + \text{text tokens})^2,$$

where the attention entries count the scores in one dense attention matrix, for one head in one layer. A 336 × 336 image with 14-pixel patches gives $24 \times 24 = 576$ tokens, the number LLaVA-1.5 passes to its language model for each image. Eight such frames give 4,608 tokens before any text. Some models add special tokens per image or compress tokens with a resampler; count what *your* model actually does.

In [ ]:
for side, patch in ((224, 16), (224, 14), (336, 14)):
    print(f"{side} x {side} image, {patch}-pixel patches: {(side // patch) ** 2} tokens per frame")

### ✍️ Your turn: `visual_token_budget`

Write `visual_token_budget(height, width, patch_size, frames=1, text_tokens=0)`.

- return a dictionary with four integers: `tokens_per_frame` $= (H // P)(W // P)$, `visual_tokens` $=$ frames × tokens per frame, `total_tokens` $=$ visual + text tokens, and `attention_entries` $=$ total tokens squared.

*Hint:* use integer division `//`, as a patch grid only counts whole patches.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def visual_token_budget_reference(height, width, patch_size, frames=1, text_tokens=0):
    """Token counts for `frames` images of height x width cut into square patches.

    Returns a dict: tokens_per_frame, visual_tokens, total_tokens (visual + text) and
    attention_entries, the size of one dense attention score matrix (total_tokens squared).
    """
    per_frame = (height // patch_size) * (width // patch_size)
    visual = per_frame * frames
    total = visual + text_tokens
    return {"tokens_per_frame": per_frame, "visual_tokens": visual, "total_tokens": total,
            "attention_entries": total * total}

In [ ]:
def visual_token_budget(height, width, patch_size, frames=1, text_tokens=0):
    """Token counts for `frames` images of height x width cut into square patches.

    Returns a dict: tokens_per_frame, visual_tokens, total_tokens (visual + text) and
    attention_entries, the size of one dense attention score matrix (total_tokens squared).
    """
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
visual_token_budget = practice("visual_token_budget", visual_token_budget, visual_token_budget_reference, cases=[
    (16, 16, 4),
    (336, 336, 14, 8, 64),
    (224, 224, 16, 1, 0),
    (8, 8, 8, 3, 2),  # edge case: one patch per frame
])

In [ ]:
print(f"{'setting':40s} {'visual':>8s} {'total':>8s} {'attention entries':>18s}")
for label, args in (("our image, 4-pixel patches", (16, 16, 4, 1, 8)),
                    ("our image, 8 frames", (16, 16, 4, 8, 8)),
                    ("336 x 336, 14-pixel patches", (336, 336, 14, 1, 64)),
                    ("336 x 336, 14-pixel patches, 8 frames", (336, 336, 14, 8, 64)),
                    ("672 x 672, 14-pixel patches", (672, 672, 14, 1, 64))):
    budget = visual_token_budget(*args)
    print(f"{label:40s} {budget['visual_tokens']:8,d} {budget['total_tokens']:8,d} {budget['attention_entries']:18,d}")

**Expected behaviour.** ✅ on four cases. One of our images costs 16 patch tokens; eight frames of 336 × 336 cost 4,608 visual tokens, and with 64 text tokens one attention matrix has about 21.8 million entries, more than fifty times as many as for a single image. Doubling the resolution to 672 × 672 quadruples the visual tokens again. Resolution, number of frames and context length compete for the same budget, so choose them for the task: reading small print needs resolution, a question about motion needs frames.

### Frames are samples

A video model sees only the frames it is given. **Uniform sampling** spreads $k$ frames evenly over the clip: it covers the whole duration, but an event shorter than the gap between two samples can fall in between. Whatever happens between sampled frames is simply not evidence the model has.

### ✍️ Your turn: `uniform_frame_indices`

Write `uniform_frame_indices(n_frames, n_samples)`: return an integer array of `n_samples` frame numbers spread evenly from frame 0 to frame `n_frames - 1` inclusive, rounded **down**.

*Hint:* `np.linspace(0, n_frames - 1, n_samples)` gives the evenly spaced positions; `.astype(int)` rounds non-negative numbers down.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def uniform_frame_indices_reference(n_frames, n_samples):
    """n_samples frame numbers spread evenly from the first frame to the last, rounded down."""
    return np.linspace(0, n_frames - 1, n_samples).astype(int)

In [ ]:
def uniform_frame_indices(n_frames, n_samples):
    """n_samples frame numbers spread evenly from the first frame to the last, rounded down."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
uniform_frame_indices = practice("uniform_frame_indices", uniform_frame_indices, uniform_frame_indices_reference, cases=[
    (32, 8),
    (10, 4),
    (5, 1),  # edge case: a single sample is the first frame
])
print(uniform_frame_indices(32, 8))

### 🧪 Try it: a brief event in a 32-frame clip

The clip below shows a large blue horizontal bar moving slowly to the right. In frames 18 and 19, and only there, the bar turns red. The cell samples `n_samples` frames uniformly, asks the model "what colour is the bar ?" about each sampled frame, and reports whether the red event was seen and what the sampled frames cost in visual tokens, counted both as one embedding token per frame (our model) and as 16 patch tokens per frame.

In [ ]:
n_samples = 8  # try 4, 16 and 32, then run this cell again

n_frames, event = 32, (18, 19)
clip = {"colour": np.array([0 if t in event else 2 for t in range(n_frames)]),  # red during the event, blue otherwise
        "size": np.ones(n_frames, dtype=int), "orientation": np.zeros(n_frames, dtype=int),
        "row": np.full(n_frames, 6), "col": np.arange(n_frames) // 8, "noise_seed": 500 + np.arange(n_frames)}
sampled = uniform_frame_indices(n_frames, n_samples)
sampled_scenes = {key: value[sampled] for key, value in clip.items()}
answers, _ = ask(model, sampled_scenes, "colour", None, mean, std)
seen_event = bool(np.isin(sampled, event).any())
print("sampled frames:", sampled.tolist())
print("answers:       ", [LM_VOCAB[a] for a in answers])
print(f"event seen: {seen_event}; visual tokens: {n_samples * model['n_visual']} as one embedding per frame, "
      f"{visual_token_budget(16, 16, 4, n_samples)['visual_tokens']} as patch tokens")
fig, axes = plt.subplots(1, len(sampled), figsize=(1.1 * len(sampled), 1.5), squeeze=False)
for ax, t, image in zip(axes[0], sampled, render(sampled_scenes)):
    ax.imshow(image, interpolation="nearest")
    ax.set_title(str(t), fontsize=7)
    ax.axis("off")
plt.show()

caught = {k: bool(np.isin(uniform_frame_indices(n_frames, k), event).any()) for k in (4, 8, 16, 32)}
print("event seen with 4, 8, 16, 32 samples:", caught)
assert caught == {4: False, 8: False, 16: True, 32: True}

**Expected behaviour.** With 8 samples the frames are 0, 4, 8, 13, 17, 22, 26 and 31: the event in frames 18 and 19 falls between two samples, every answer is "blue", and no reasoning afterwards could recover the red frames. With 4 samples it is missed as well; with 16 samples frame 18 is included and the model reports red there, at twice the token cost of 8. A two-frame event is only *guaranteed* to be caught when no gap between neighbouring samples is longer than one frame. Exercise 1 makes that precise. Real video models also need training on ordered frames to answer questions about time, which this image model has never had.

## Part 5 · The capstone scaffold

The capstone asks for a small multimodal project whose claims match its evidence. Start from this notebook's model or extend it: a new attribute (a fifth colour, a new shape), a new question type, or a two-frame question such as "did the colour change?". Whatever you build, the report must make these things checkable:

1. **the task and the claims you do not make**: retrieval, generation, OCR and temporal reasoning are different capabilities;
2. **the data**: the manifest, the split unit and proof that evaluation sources are independent of training sources;
3. **the input path**: normalisation constants, patch size, visual tokens per image, vocabulary and text length;
4. **the budget and the trainable modules**: tokens per example and which parts were frozen;
5. **the results with grounding controls**: accuracy by slice, an image-swap or counterfactual test, and an abstention or hallucination check;
6. **failures and limits**: examples of mistakes and the conditions you did not test.

The next cell assembles such a report for this notebook's model, from the numbers measured above, and `check_capstone_report` tests that nothing is missing. For your capstone, change the model and the data, keep the checks, and add the evidence your extension needs.

In [ ]:
swap = np.random.default_rng(1).permutation(len(held_out_patches))
held_out_types = held_out_vqa[2]
capstone_report = {
    "task": "Answer three fixed questions (colour, size, orientation) about one bar in a generated 16 x 16 image.",
    "claims_not_made": ["natural photographs", "reading text in images", "several objects or counting",
                        "open-ended questions", "reasoning over time in video"],
    "data": {"generator": "sample_scenes and render with seeds 101, 202 and 303",
             "split_unit": "attribute combination",
             "train": {"images": len(train_images), "combinations": len(TRAIN_COMBOS)},
             "held_out_test": {"images": len(held_out_images), "combinations": len(HELD_OUT_COMBOS)},
             "overlap": sorted(set(combo_ids(train_scenes).tolist()) & set(combo_ids(held_out_scenes).tolist()))},
    "input_path": {"mean": mean.round(4).tolist(), "std": std.round(4).tolist(), "patch_size": 4,
                   "visual_input": "aligned image embedding", "text_length": TEXT_LENGTH, "vocabulary": len(LM_VOCAB)},
    "token_budget": {"visual_tokens": model["n_visual"], "text_tokens": TEXT_LENGTH - 1,
                     "sequence_length": model["n_visual"] + TEXT_LENGTH - 1},
    "trainable_modules": {"vision encoder": "frozen", "projector": "trained", "language model": "trained from scratch"},
    "results": {"seen": vqa_report(model, seen_patches, *seen_vqa), "held_out": vqa_report(model, held_out_patches, *held_out_vqa)},
    "grounding_controls": {"image_swap_colour_accuracy": vqa_report(model, held_out_patches[swap], *held_out_vqa)["colour"],
                           "counterfactual_both_correct": {key: value["both_correct"] for key, value in grounding.items()},
                           "blank_images_answered_none": 1 - invented},
    "failures": [],
    "limitations": ["16 x 16 drawings of one bar on a plain background", "three question templates, single-word answers",
                    "a language model trained from scratch on 800 questions", "one seed per configuration in this notebook"],
}
predicted, _ = answer_questions(model, held_out_patches, held_out_vqa[0], held_out_vqa[1])
for i in np.flatnonzero(predicted != held_out_vqa[3])[:3]:
    capstone_report["failures"].append(f"{describe(held_out_scenes)[i]}: asked '{decode(held_out_vqa[0][i][:held_out_vqa[1][i]])}', "
                                       f"answered '{LM_VOCAB[predicted[i]]}'")


def check_capstone_report(report):
    required = ("task", "claims_not_made", "data", "input_path", "token_budget", "trainable_modules",
                "results", "grounding_controls", "failures", "limitations")
    missing = [key for key in required if key not in report]
    assert not missing, f"the report is missing: {missing}"
    assert report["data"]["overlap"] == [], "evaluation combinations must not appear in training"
    for split in report["results"].values():
        assert all(0.0 <= value <= 1.0 for value in split.values()), "results must be fractions"
    assert len(report["grounding_controls"]) >= 2, "include at least two grounding controls"
    assert len(report["limitations"]) >= 3 and report["claims_not_made"], "state the limits and the claims you do not make"
    print("The capstone report has every section.")


check_capstone_report(capstone_report)
for key, value in capstone_report.items():
    print(f"{key}: {value}")

**Expected behaviour.** The check passes and the report prints section by section: no overlap between training and held-out combinations, the input path, a budget of one visual token plus eight text tokens, the trainable modules, results for seen and held-out combinations, three grounding controls and the first held-out failures. Every number in it was measured in this notebook, and the limitations say what was not. When you extend the project, the report should grow with it: new slices for the new attribute, a counterfactual for it, and an honest list of what remains untested.

## Exercises

1. **✍️ `largest_gap`.** Write a function that returns the most consecutive frames that lie strictly between two neighbouring sampled frames. An event that lasts longer than that gap can never be missed when the samples include the first and last frames. Use it to find the smallest number of uniform samples that guarantees seeing a two-frame event in a 32-frame clip.
2. **Slices by combination.** Use `accuracy_by_group` with `combo_ids(held_out_scenes)` as the groups. Which held-out combination is hardest, and does the counterfactual table point to the same attribute?
3. **A POPE-style probe.** Add the yes/no question "is there a red bar ?" and ask it about images whose bar is another colour and about blank images. A model that says "yes" too often is hallucinating the object. How balanced must the training answers be to avoid that?
4. **Your capstone.** Add a fifth colour, retrain the whole pipeline, and extend `capstone_report` with its slices and counterfactuals.

### ✍️ Exercise 1: `largest_gap`

Write `largest_gap(indices)`: for sorted, distinct frame numbers, return the largest number of frames strictly between two neighbouring samples, as an `int`, and 0 when fewer than two frames are sampled. For the samples `[0, 4, 8]` the gaps hold frames 1–3 and 5–7, so the answer is 3.

*Hint:* `np.diff` gives the distances between neighbours; a distance of $d$ leaves $d - 1$ frames unseen.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def largest_gap_reference(indices):
    """The most consecutive frames that lie strictly between two neighbouring sampled frames.

    indices: sorted, distinct frame numbers. Returns 0 when fewer than two frames are sampled.
    """
    indices = np.asarray(indices)
    if len(indices) < 2:
        return 0
    return int((np.diff(indices) - 1).max())

In [ ]:
def largest_gap(indices):
    """The most consecutive frames that lie strictly between two neighbouring sampled frames.

    indices: sorted, distinct frame numbers. Returns 0 when fewer than two frames are sampled.
    """
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
largest_gap = practice("largest_gap", largest_gap, largest_gap_reference, cases=[
    (np.array([0, 4, 8]),),
    (uniform_frame_indices(32, 8),),
    (np.array([7]),),  # edge case: one sample has no gaps
])
needed = next(k for k in range(2, 33) if largest_gap(uniform_frame_indices(32, k)) < 2)
print(f"largest gap with 8 samples: {largest_gap(uniform_frame_indices(32, 8))} frames; "
      f"{needed} uniform samples guarantee seeing any two-frame event in 32 frames")

**Expected behaviour.** ✅, a largest gap of 4 frames for 8 samples, and 17 samples as the smallest number that leaves no gap of two frames. With 16 samples the last gap, between frames 28 and 31, still hides frames 29 and 30: the 🧪 cell caught the event at frames 18 and 19 partly by luck of where it happened. Guaranteeing that a brief event is seen costs many more frames, and therefore tokens, than covering the clip's duration.

## Common errors and debugging

- **One number for everything.** An aggregate accuracy can hide a slice that fails. Report slices that match the intended use, with the number of examples in each.
- **Counterfactuals that change two things.** If changing the size also moves the bar or redraws the noise, an answer change no longer isolates the attribute. Keep everything else fixed, as the scene record makes possible here.
- **Testing only on data with the same coincidences.** A shortcut that holds in training usually holds in a test set collected the same way. Counterfactual pairs, held-out combinations and text-only or image-swap baselines are how you find it.
- **Treating confidence as evidence.** The blank-image answers came with probabilities near 1. Calibrated confidence must be measured, not assumed.
- **Budgeting only the image.** Frames, crops and tiles multiply the visual tokens, and attention cost grows with the square of the total sequence. Count text tokens too.
- **Claiming video understanding from frames.** A model that sees unordered samples cannot be trusted about events between them or about their order unless it was trained and tested on those questions.

## Summary

- Slices show where a model fails; counterfactual images show whether its answers follow the evidence they should depend on.
- A model trained on confounded data scored 1.0 on a matching test set and near 0 on size counterfactuals: it read colour, not size.
- Without unanswerable examples the model invented answers for empty images with high confidence; 10% blank training images with the answer `none` fixed that case, while held-out accuracy on real images stayed in a similar range.
- Visual tokens scale with resolution and frames, attention with the square of the sequence, and uniform frame sampling can miss brief events entirely.
- A capstone report ties every claim to a measurement, and lists what was not tested.

## Next

This is the last notebook of the Vision-Language Models course. Complete the capstone on the course page, and then continue with **Voice & Speech Models**, which applies the same habits to audio: careful data, explicit alignment between two modalities, and evaluation that tests the evidence a model actually uses, for speech recognition and speech synthesis.